In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, average_precision_score

import xgboost as xgb


In [ ]:
# Load processed datasets
fraud_df = pd.read_csv("../data/processed/fraud_data_final.csv")
creditcard_df = pd.read_csv("../data/processed/creditcard_processed.csv")

print("Fraud data shape:", fraud_df.shape)
print("Credit card data shape:", creditcard_df.shape)


In [ ]:
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_recall_curve, auc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
# --- 1. Load preprocessed dataset ---
import pandas as pd

fraud_df = pd.read_csv("../data/processed/fraud_data_final.csv")

# --- 2. Separate features and target ---
X = fraud_df.drop(columns=["class"])
y = fraud_df["class"]

# --- 3. Stratified train-test split ---
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# --- 4. Quick check ---
print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)
print("Class distribution in train set:\n", y_train.value_counts())


In [ ]:
# --- 5. Baseline Logistic Regression ---
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve, auc

# Initialize Logistic Regression
logreg = LogisticRegression(max_iter=1000, random_state=42)

# Fit on training data
logreg.fit(X_train, y_train)

# Predict on test data
y_pred = logreg.predict(X_test)
y_proba = logreg.predict_proba(X_test)[:, 1]

# --- 6. Evaluate performance ---
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

# Calculate AUC-PR
precision, recall, _ = precision_recall_curve(y_test, y_proba)
auc_pr = auc(recall, precision)
print(f"AUC-PR: {auc_pr:.4f}")


In [ ]:
# --- Cell 3: Ensemble Model (XGBoost) ---

from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, average_precision_score

# Instantiate XGBoost model (basic hyperparameters, can tune later)
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

# Train model
xgb_model.fit(X_train, y_train)

# Predict on test set
y_pred = xgb_model.predict(X_test)
y_prob = xgb_model.predict_proba(X_test)[:, 1]  # for AUC-PR

# Evaluate
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("AUC-PR: {:.4f}".format(average_precision_score(y_test, y_prob)))


In [ ]:
from xgboost import plot_importance
import matplotlib.pyplot as plt

plt.figure(figsize=(12,8))
plot_importance(xgb_model.get_booster(), max_num_features=15, importance_type='weight')
plt.title("Top 15 Feature Importances - XGBoost")
plt.show()


In [ ]:
# --- Stratified K-Fold Cross-Validation for XGBoost ---
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_metrics = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    # Initialize XGBoost classifier
    xgb_model = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        n_estimators=100,
        max_depth=5,
        random_state=42,
        use_label_encoder=False  # avoids warning
    )
    
    # Train the model
    xgb_model.fit(X_tr, y_tr)
    
    # Predict probabilities and labels
    y_pred_proba = xgb_model.predict_proba(X_val)[:, 1]
    y_pred = xgb_model.predict(X_val)
    
    # Metrics
    report = classification_report(y_val, y_pred, output_dict=True)
    cm = confusion_matrix(y_val, y_pred)
    precision, recall, _ = precision_recall_curve(y_val, y_pred_proba)
    auc_pr = auc(recall, precision)
    
    print(f"\n--- Fold {fold} ---")
    print("Classification Report:")
    print(classification_report(y_val, y_pred))
    print("Confusion Matrix:\n", cm)
    print(f"AUC-PR: {auc_pr:.4f}")
    
    # Store metrics
    fold_metrics.append({
        "fold": fold,
        "f1_macro": report['macro avg']['f1-score'],
        "accuracy": report['accuracy'],
        "auc_pr": auc_pr
    })

# Summary of folds
summary_df = pd.DataFrame(fold_metrics)
print("\n--- Cross-Validation Summary ---")
print(summary_df.describe())


In [ ]:
import joblib
# Save the Fraud model IMMEDIATELY before it gets overwritten
joblib.dump(xgb_model, "../models/xgb_fraud_data_model.pkl")
print("Fraud model saved successfully!")

?? Model Comparison and Selection

We trained and evaluated two models for fraud detection:

1. Logistic Regression (Baseline Model)

Strengths:

Simple and highly interpretable

Fast to train

Weaknesses:

Very poor recall and F1-score for the fraud class

Failed to capture complex, non-linear patterns

Performance Summary:

AUC-PR ? 0.50

F1-score for fraud class ? 0.00

Indicates performance close to random guessing

This model is useful only as a baseline and is not suitable for deployment in a real fraud detection system.

2. XGBoost (Ensemble Model)

Strengths:

Captures complex non-linear relationships

Handles high-dimensional data effectively

Strong performance on imbalanced datasets

Weaknesses:

Less interpretable than Logistic Regression

Higher computational cost

Performance Summary:

Mean AUC-PR (5-fold CV): ? 0.984

Mean F1-score (macro): ? 0.966

Stable performance across folds (very low variance)

XGBoost significantly outperformed the baseline model across all evaluation metrics.

?? Final Model Selection

XGBoost is selected as the final model for fraud detection.

Justification:

It achieves a much higher AUC-PR, which is critical for imbalanced fraud data.

High recall for fraudulent transactions reduces false negatives, minimizing financial loss.

Cross-validation results show consistent and reliable performance.

Although less interpretable, its superior predictive power justifies its use, especially when combined with explainability tools such as SHAP (addressed in Task 3)

In [ ]:
# --- Baseline Logistic Regression for Credit Card Data ---

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, auc
import pandas as pd

# Load credit card data
cc_data =pd.read_csv("../data/processed/creditcard_processed.csv")


# Separate features and target
X_cc = cc_data.drop(columns=["Class"])
y_cc = cc_data["Class"]

# Train-test split with stratification
X_cc_train, X_cc_test, y_cc_train, y_cc_test = train_test_split(
    X_cc, y_cc, test_size=0.2, random_state=42, stratify=y_cc
)

# Scale features
scaler_cc = StandardScaler()
X_cc_train_scaled = scaler_cc.fit_transform(X_cc_train)
X_cc_test_scaled = scaler_cc.transform(X_cc_test)

# Train Logistic Regression baseline
lr_cc = LogisticRegression(random_state=42, max_iter=1000)
lr_cc.fit(X_cc_train_scaled, y_cc_train)

# Predict and evaluate
y_cc_pred = lr_cc.predict(X_cc_test_scaled)
y_cc_proba = lr_cc.predict_proba(X_cc_test_scaled)[:, 1]

print("Classification Report:")
print(classification_report(y_cc_test, y_cc_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_cc_test, y_cc_pred))

# Compute AUC-PR
precision, recall, _ = precision_recall_curve(y_cc_test, y_cc_proba)
auc_pr_cc = auc(recall, precision)
print(f"AUC-PR: {auc_pr_cc:.4f}")


In [ ]:
import pandas as pd

# Load the creditcard dataset
creditcard_df = pd.read_csv("../data/processed/creditcard_processed.csv")


# Quick check
creditcard_df.head()
creditcard_df.info()


In [ ]:
from sklearn.model_selection import train_test_split

# Assuming your creditcard dataframe is named creditcard_df
X_cc = creditcard_df.drop('Class', axis=1)
y_cc = creditcard_df['Class']

# Stratified split to preserve class distribution
X_train_cc, X_test_cc, y_train_cc, y_test_cc = train_test_split(
    X_cc, y_cc, test_size=0.2, random_state=42, stratify=y_cc
)

# Check shapes
print("Training set shape:", X_train_cc.shape)
print("Test set shape:", X_test_cc.shape)
print("Class distribution in train set:\n", y_train_cc.value_counts())


In [ ]:
import xgboost as xgb

# Initialize XGBoost classifier
xgb_cc_model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    n_estimators=100,
    max_depth=5,
    random_state=42
)

# Train the model
xgb_cc_model.fit(X_train_cc, y_train_cc)

# Predict and evaluate
y_cc_pred_xgb = xgb_cc_model.predict(X_test_cc)
y_cc_proba_xgb = xgb_cc_model.predict_proba(X_test_cc)[:, 1]

print("Classification Report:")
print(classification_report(y_test_cc, y_cc_pred_xgb))
print("Confusion Matrix:")
print(confusion_matrix(y_test_cc, y_cc_pred_xgb))

# Compute AUC-PR
precision, recall, _ = precision_recall_curve(y_test_cc, y_cc_proba_xgb)
auc_pr_cc_xgb = auc(recall, precision)
print(f"AUC-PR: {auc_pr_cc_xgb:.4f}")


In [ ]:
# Save the Credit Card model
joblib.dump(xgb_model, "../models/xgb_creditcard_model.pkl")
print("Credit Card model saved successfully!")

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score, precision_recall_curve, auc
import numpy as np


In [ ]:
# Initialize StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Lists to store metrics per fold
f1_scores = []
accuracies = []
auc_prs = []

# Features and target for credit card dataset
X = X_train_cc
y = y_train_cc

# Iterate through folds
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f"--- Fold {fold} ---")
    
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # Initialize XGBoost model
    xgb_model = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        n_estimators=100,
        max_depth=5,
        random_state=42
    )
    
    # Fit the model
    xgb_model.fit(X_tr, y_tr)
    
    # Predict
    y_pred = xgb_model.predict(X_val)
    y_pred_proba = xgb_model.predict_proba(X_val)[:, 1]
    
    # Metrics
    f1 = f1_score(y_val, y_pred, average='macro')
    acc = accuracy_score(y_val, y_pred)
    precision, recall, _ = precision_recall_curve(y_val, y_pred_proba)
    auc_pr = auc(recall, precision)
    
    # Store metrics
    f1_scores.append(f1)
    accuracies.append(acc)
    auc_prs.append(auc_pr)
    
    # Print fold metrics
    print("Classification Report:")
    print(classification_report(y_val, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_val, y_pred))
    print(f"AUC-PR: {auc_pr:.4f}\n")

# Cross-validation summary
print("--- Cross-Validation Summary ---")
print(f"F1-score (macro): Mean={np.mean(f1_scores):.4f}, Std={np.std(f1_scores):.4f}")
print(f"Accuracy: Mean={np.mean(accuracies):.4f}, Std={np.std(accuracies):.4f}")
print(f"AUC-PR: Mean={np.mean(auc_prs):.4f}, Std={np.std(auc_prs):.4f}")


In [ ]:
import joblib

# Save the trained XGBoost model
joblib.dump(xgb_model, "../models/xgb_creditcard_model.pkl")


In [ ]:
# Save the trained XGBoost model for Fraud_Data
joblib.dump(xgb_model, "../models/xgb_fraud_data_model.pkl")


In [ ]:
[x for x in globals() if "xgb" in x.lower()]
